# Video to anime: cloud stage on Kaggle (AnimateDiff + lineart ControlNet, IP-Adapter optional)

Setup before running: upload `work/cloud/bundle_<shot>.zip` (made by `scripts/make_cloud_bundle.py`) as a Kaggle **dataset** (Kaggle unzips it automatically; the notebook handles either form), add it to this notebook, and set **Accelerator = GPU (T4 recommended)** and **Internet = On** (Settings panel; Internet needs a phone-verified account). Then Run All.
Results land in `/kaggle/working/result_<shot>.zip`. Unzip it at the project root locally so the folders land in `work/default/`.

In [ ]:
# 1. Get the project files into /kaggle/working/project.
# Kaggle usually UNZIPS an uploaded dataset, so look for the extracted project (cli.py) first, then for a zip.
import glob, shutil, zipfile, pathlib
PROJECT = pathlib.Path('/kaggle/working/project')
SHOT = None   # set e.g. SHOT = 'dance_run1' only if the frames folder holds several shots
shutil.rmtree(PROJECT, ignore_errors=True)
clis = [p for p in sorted(glob.glob('/kaggle/input/**/cli.py', recursive=True)) if (pathlib.Path(p).parent / 'pipeline').is_dir()]
zips = sorted(glob.glob('/kaggle/input/**/bundle_*.zip', recursive=True))
if clis:
    shutil.copytree(pathlib.Path(clis[0]).parent, PROJECT)
    print('copied extracted dataset from', pathlib.Path(clis[0]).parent)
elif zips:
    zipfile.ZipFile(zips[-1]).extractall(PROJECT)
    print('unzipped', zips[-1])
else:
    print('Found under /kaggle/input:', glob.glob('/kaggle/input/*'))
    raise SystemExit('No project found under /kaggle/input. Add the bundle dataset: Add Input > your dataset.')
shots = sorted(d.name for d in (PROJECT / 'work/default/frames').iterdir() if d.is_dir())
assert shots, 'no frames folder in the bundle (work/default/frames/<shot>)'
SHOT = SHOT or shots[0]
print('shot:', SHOT, '| frames:', len(list((PROJECT / 'work/default/frames' / SHOT).glob('frame_*.png'))))

In [ ]:
# 2. ComfyUI + custom nodes + python deps (about 3 minutes)
%cd /kaggle/working/project
!git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git third_party/ComfyUI
!pip install -q -r third_party/ComfyUI/requirements.txt
!pip install -q requests pyyaml typer tqdm pillow opencv-python-headless scikit-image huggingface-hub
!cd third_party/ComfyUI/custom_nodes && git clone --depth 1 https://github.com/Kosinkadink/ComfyUI-AnimateDiff-Evolved.git
!cd third_party/ComfyUI/custom_nodes && git clone --depth 1 https://github.com/cubiq/ComfyUI_IPAdapter_plus.git
!cd third_party/ComfyUI/custom_nodes && git clone --depth 1 https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git
!cd third_party/ComfyUI/custom_nodes && git clone --depth 1 https://github.com/Kosinkadink/ComfyUI-Advanced-ControlNet.git
!cd third_party/ComfyUI/custom_nodes && git clone --depth 1 https://github.com/Fannovel16/comfyui_controlnet_aux.git
!pip install -q -r third_party/ComfyUI/custom_nodes/ComfyUI-VideoHelperSuite/requirements.txt
# only the controlnet_aux deps anime lineart needs (the full list pulls mediapipe, which can break numpy)
!pip install -q python-dateutil omegaconf addict yacs matplotlib scikit-learn einops timm

In [ ]:
# 3. Models: checkpoint, ControlNet, lineart extractor, AnimateDiff motion module (and IP-Adapter, only if the config enables it)
!python scripts/download_models.py --cloud

In [ ]:
# 4. Start a FRESH ComfyUI in the background (a server left over from an earlier run does not know nodes installed since)
import subprocess, time, requests, os, pathlib
subprocess.run(['pkill', '-f', 'main.py --listen'])   # stop any ComfyUI this notebook started before
time.sleep(3)
nodes = pathlib.Path('third_party/ComfyUI/custom_nodes')
absent = [d for d in ('ComfyUI-AnimateDiff-Evolved', 'ComfyUI_IPAdapter_plus', 'ComfyUI-VideoHelperSuite',
                      'ComfyUI-Advanced-ControlNet', 'comfyui_controlnet_aux') if not (nodes / d).is_dir()]
assert not absent, f'custom nodes not installed: {absent}. Run step 2 again (the updated notebook installs them).'
env = dict(os.environ, CUDA_VISIBLE_DEVICES='0')
log = open('/kaggle/working/comfyui.log', 'w')
server = subprocess.Popen(['python', 'main.py', '--listen', '127.0.0.1', '--port', '8188'],
                          cwd='third_party/ComfyUI', stdout=log, stderr=subprocess.STDOUT, env=env)
for _ in range(180):
    try:
        if requests.get('http://127.0.0.1:8188/system_stats', timeout=2).ok: break
    except requests.RequestException: time.sleep(2)
else:
    print(open('/kaggle/working/comfyui.log').read()[-3000:])
    raise RuntimeError('ComfyUI did not start; the end of its log is printed above')
print('ComfyUI is up')
# the custom nodes must have loaded, or the workflow will be rejected
info = requests.get('http://127.0.0.1:8188/object_info').json()
missing = [n for n in ('VHS_LoadImagesPath', 'ADE_AnimateDiffLoaderGen1', 'ADE_StandardUniformContextOptions',
                       'IPAdapterUnifiedLoader', 'IPAdapter', 'AnimeLineArtPreprocessor',
                       'ACN_ControlNetLoaderAdvanced', 'ACN_AdvancedControlNetApply_v2') if n not in info]
print('missing nodes:', missing or 'none')
if missing:
    print([l for l in open('/kaggle/working/comfyui.log').read().splitlines() if 'IMPORT FAILED' in l or 'Cannot import' in l or 'Error' in l][-15:])
assert not missing, 'a custom node failed to load; the lines above (or /kaggle/working/comfyui.log) say why'

In [ ]:
# 5. Stylize the whole shot in one AnimateDiff pass (with IP-Adapter if the config enables it) (about 5-15 minutes for 48 frames)
!python cli.py animatediff {SHOT}

In [ ]:
# 6. Measure flicker and run the deflicker on top (CPU). Compare with the per-frame baseline of the same shot (for dance_run1: static change 5.85, color flicker 1.86)
!python cli.py flicker {SHOT} --stage animatediff
!python cli.py encode {SHOT} --stage animatediff
!python cli.py deflicker {SHOT} --source animatediff
!python cli.py flicker {SHOT} --stage deflicker_animatediff

In [ ]:
# 7. Package the results (frames, settings records, and the two mp4s) for download
import shutil
out = pathlib.Path('/kaggle/working')
keep = PROJECT / 'work/default'
tmp = out / 'pack'; shutil.rmtree(tmp, ignore_errors=True)
for sub in ('animatediff', 'deflicker_animatediff'):
    shutil.copytree(keep / sub / SHOT, tmp / 'work/default' / sub / SHOT)
for mp4 in (keep / 'out').glob(f'{SHOT}_*.mp4'):
    (tmp / 'work/default/out').mkdir(parents=True, exist_ok=True); shutil.copy(mp4, tmp / 'work/default/out' / mp4.name)
shutil.make_archive(str(out / f'result_{SHOT}'), 'zip', tmp)
shutil.rmtree(tmp)
server.terminate()
print('download result_%s.zip from the Output tab' % SHOT)